In [1]:
import os
import music_scraper  

if not os.path.exists("artist_metadata_clean.csv"):
    print("No existing metadata found — starting scrape...")
    music_scraper.run()  
else:
    print("artist_metadata_clean.csv already exists — skipping scrape ✓")
    print("(Delete artist_metadata_clean.csv and re-run this cell to re-scrape)")

No existing metadata found — starting scrape...
Scraping 154 unique primary artists (from top 200 entries)...
Top 5 preview: ['Sam Smith', 'Bizarrap', 'Manuel Turizo', 'Bad Bunny', 'Joji']

Scraping: Sam Smith
  ✓ Wikipedia (325 chars)
Scraping: Bizarrap
  ✓ Wikipedia (374 chars)
Scraping: Manuel Turizo
  ✓ Wikipedia (399 chars)
Scraping: Bad Bunny
  ✓ Wikipedia (329 chars)
Scraping: Joji
  ✗ Last.fm bio failed music validation — discarding
  ✗ No valid music bio found
Scraping: Beyoncé
  ✓ Wikipedia (369 chars)
Scraping: Harry Styles
  ✓ Wikipedia (641 chars)
Scraping: Rema
  ✓ Wikipedia (489 chars)
Scraping: Rauw Alejandro
  ✓ Wikipedia (446 chars)
Scraping: Drake
  ✓ Wikipedia (539 chars)
Scraping: Luar La L
  ✗ No valid music bio found
Scraping: The Weeknd
  ✓ Wikipedia (503 chars)
Scraping: Ruth B.
  ✓ Wikipedia (501 chars)
Scraping: Shakira
  ✓ Wikipedia (521 chars)
Scraping: Elley Duhé
  ✓ Last.fm (1486 chars)
Scraping: Beach Weather
  ✓ Wikipedia (202 chars)
Scraping: Billie Ei

In [2]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def build_ir_engine(metadata_path="artist_metadata_clean.csv"):
  
    df = pd.read_csv(metadata_path)
    
    df['bio_clean'] = df['bio_clean'].fillna("")
    
    vectorizer = TfidfVectorizer(
        stop_words='english',
        max_features=5000, 
        sublinear_tf=True
    )
    
    tfidf_matrix = vectorizer.fit_transform(df['bio_clean'])
    
    cosine_sim = cosine_similarity(tfidf_matrix)
    
    return df, tfidf_matrix, cosine_sim

def get_text_recommendations(artist_name, df, sim_matrix, top_n=5):
    try:
        idx = df[df['artist'].str.lower() == artist_name.lower()].index[0]
        
        sim_scores = list(enumerate(sim_matrix[idx]))
        
        sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
        
        top_indices = [i[0] for i in sim_scores[1:top_n+1]]
        
        return df['artist'].iloc[top_indices].tolist()
    except IndexError:
        return ["Artist not found in metadata."]

if __name__ == "__main__":
    print("Building Vector Space Model...")
    df_meta, tfidf, sim_matrix = build_ir_engine()
    
    test_artist = df_meta['artist'].iloc[0] 
    print(f"\nTop 5 Text-Based Recommendations for '{test_artist}':")
    recs = get_text_recommendations(test_artist, df_meta, sim_matrix)
    for i, name in enumerate(recs, 1):
        print(f"{i}. {name}")

    np.save("baseline_similarity.npy", sim_matrix)
    print("\nSimilarity matrix saved as 'baseline_similarity.npy'")

Building Vector Space Model...

Top 5 Text-Based Recommendations for 'Sam Smith':
1. Joel Corry
2. James Arthur
3. Shubh
4. Lil Tjay
5. Lewis Capaldi

Similarity matrix saved as 'baseline_similarity.npy'


In [3]:
import pandas as pd
import numpy as np
import networkx as nx

def build_music_graph(metadata_path, songs_path, similarity_path,
                      sim_threshold=0.12):
  
    df_meta  = pd.read_csv(metadata_path)   
    df_songs = pd.read_csv(songs_path)      
    sim_matrix = np.load(similarity_path)

    df_songs["primary_artist"] = df_songs["artists"].str.split(";").str[0].str.strip()
    artist_genres = (
        df_songs.groupby("primary_artist")["track_genre"]
        .apply(lambda x: set(x.dropna()))
        .to_dict()
    )

    G = nx.Graph()

    for i, row in df_meta.iterrows():
        G.add_node(i, artist=row["artist"])

    n = len(df_meta)

    for i in range(n):
        for j in range(i + 1, n):
            artist_i = df_meta["artist"].iloc[i]
            artist_j = df_meta["artist"].iloc[j]

            genres_i = artist_genres.get(artist_i, set())
            genres_j = artist_genres.get(artist_j, set())

            edge_added = False

            if genres_i & genres_j:
                G.add_edge(i, j, weight=0.6)
                edge_added = True

            if sim_matrix[i][j] > sim_threshold:
                if G.has_edge(i, j):
                    G[i][j]["weight"] = min(1.0, G[i][j]["weight"] + sim_matrix[i][j])
                else:
                    G.add_edge(i, j, weight=float(sim_matrix[i][j]))

    return G, df_meta, sim_matrix

def hybrid_recommendation(artist_name, G, df, sim_matrix, alpha=0.6, top_n=5):
   
    try:
        idx = df[df["artist"].str.lower() == artist_name.lower()].index[0]
    except IndexError:
        return ["Artist not found in metadata."]

    scores = []
    candidates = [(u, v, p) for u, v, p in nx.jaccard_coefficient(G,
                  [(idx, j) for j in range(len(df)) if j != idx])]

    jaccard_lookup = {v: p for u, v, p in candidates}

    for j in range(len(df)):
        if j == idx:
            continue
        ir_score    = float(sim_matrix[idx][j])
        graph_score = jaccard_lookup.get(j, 0.0)
        final_score = (alpha * ir_score) + ((1 - alpha) * graph_score)
        scores.append((j, final_score))

    scores.sort(key=lambda x: x[1], reverse=True)
    top_indices = [s[0] for s in scores[:top_n]]
    return df["artist"].iloc[top_indices].tolist()

def baseline_recommendation(artist_name, df, sim_matrix, top_n=5):
    try:
        idx = df[df["artist"].str.lower() == artist_name.lower()].index[0]
    except IndexError:
        return ["Artist not found in metadata."]
    sim_scores = list(enumerate(sim_matrix[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    top_indices = [i for i, _ in sim_scores[1:top_n + 1]]
    return df["artist"].iloc[top_indices].tolist()

if __name__ == "__main__":
    print("Constructing Music Knowledge Graph...")
    G, df, sim_matrix = build_music_graph(
        metadata_path  = "artist_metadata_clean.csv",
        songs_path     = "dataset.csv",
        similarity_path= "baseline_similarity.npy"
    )
    print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges\n")

    for test_artist in ["Sam Smith", "Drake", "Bad Bunny"]:
        print(f"--- {test_artist} ---")
        baseline = baseline_recommendation(test_artist, df, sim_matrix)
        hybrid   = hybrid_recommendation(test_artist, G, df, sim_matrix)
        print(f"  Baseline : {baseline}")
        print(f"  Hybrid   : {hybrid}\n")

Constructing Music Knowledge Graph...
Graph: 154 nodes, 2514 edges

--- Sam Smith ---
  Baseline : ['Joel Corry', 'James Arthur', 'Shubh', 'Lil Tjay', 'Lewis Capaldi']
  Hybrid   : ['Akon', 'ZAYN', 'Troye Sivan', 'Camila Cabello', 'Calvin Harris']

--- Drake ---
  Baseline : ['French Montana', 'Eminem', 'Future', 'Kendrick Lamar', 'Dr. Dre']
  Hybrid   : ['The Kid LAROI', 'AP Dhillon', 'Ali Gatie', 'Lil Nas X', 'Justin Bieber']

--- Bad Bunny ---
  Baseline : ['Rels B', 'Future', 'Rauw Alejandro', 'Nicki Minaj', 'Eminem']
  Hybrid   : ['Rauw Alejandro', 'Rels B', 'El Alfa', 'Myke Towers', 'Tainy']



In [4]:
import pandas as pd
import numpy as np

def build_ground_truth(df_meta, songs_path):

    df_songs = pd.read_csv(songs_path)
    df_songs["primary_artist"] = df_songs["artists"].str.split(";").str[0].str.strip()

    artist_genres = (
        df_songs.groupby("primary_artist")["track_genre"]
        .apply(lambda x: set(x.dropna()))
        .to_dict()
    )

    known_artists = set(df_meta["artist"].tolist())
    ground_truth  = {}

    for artist in known_artists:
        genres = artist_genres.get(artist, set())
        if not genres:
            continue
        relevant = [
            other for other in known_artists
            if other != artist and (artist_genres.get(other, set()) & genres)
        ]
        if relevant:
            ground_truth[artist] = relevant

    return ground_truth


def precision_at_k(recommended, relevant, k=5):
    hits = len(set(recommended[:k]) & set(relevant))
    return hits / k


def run_full_evaluation(df_meta, sim_matrix, G, ground_truth, alpha=0.6, k=5):
    baseline_scores = []
    hybrid_scores   = []

    for artist, relevant in ground_truth.items():
        b_recs = baseline_recommendation(artist, df_meta, sim_matrix, top_n=k)
        baseline_scores.append(precision_at_k(b_recs, relevant, k))

        h_recs = hybrid_recommendation(artist, G, df_meta, sim_matrix, alpha=alpha, top_n=k)
        hybrid_scores.append(precision_at_k(h_recs, relevant, k))

    return np.mean(baseline_scores), np.mean(hybrid_scores)


if __name__ == "__main__":
    print("Building ground truth from dataset.csv genres...")
    ground_truth = build_ground_truth(df, "dataset.csv")
    print(f"Test set: {len(ground_truth)} artists with genre-based ground truth\n")

    print("Running evaluation...")
    p_baseline, p_hybrid = run_full_evaluation(
        df, sim_matrix, G, ground_truth, alpha=0.6, k=5
    )

    print("=" * 35)
    print(f"  Baseline Precision@5 : {p_baseline * 100:.1f}%")
    print(f"  Hybrid   Precision@5 : {p_hybrid   * 100:.1f}%")
    improvement = (p_hybrid - p_baseline) * 100
    print(f"  Improvement          : +{improvement:.1f}%")
    print("=" * 35)

    print("\nPer-artist sample (first 10):")
    print(f"  {'Artist':<25} {'Baseline':>10} {'Hybrid':>10}")
    print(f"  {'-'*25} {'--------':>10} {'------':>10}")
    for artist, relevant in list(ground_truth.items())[:10]:
        b = precision_at_k(baseline_recommendation(artist, df, sim_matrix), relevant)
        h = precision_at_k(hybrid_recommendation(artist, G, df, sim_matrix, alpha=0.6), relevant)
        print(f"  {artist:<25} {b*100:>9.0f}% {h*100:>9.0f}%")

Building ground truth from dataset.csv genres...
Test set: 149 artists with genre-based ground truth

Running evaluation...
  Baseline Precision@5 : 44.4%
  Hybrid   Precision@5 : 92.6%
  Improvement          : +48.2%

Per-artist sample (first 10):
  Artist                      Baseline     Hybrid
  -------------------------   --------     ------
  TOTO                             40%       100%
  Yohani                            0%       100%
  Olivia Rodrigo                   60%       100%
  Gym Class Heroes                 20%       100%
  Eminem                           80%       100%
  Myke Towers                      40%       100%
  Gabry Ponte                     100%       100%
  Train                            40%       100%
  Lady Gaga                        40%       100%
  Charlie Puth                     80%       100%
